[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/templates/47_cutmix.ipynb)

# 🟠 Medium: CutMix

**CutMix** (Yun et al. 2019) を実装する。Mixup と違って pixel 値を blend するのではなく、ある画像から矩形を **切り取って貼る**。label は paste 領域の面積比で混ぜる。

> 💡 **どこで使う**：Mixup の幾何版。ある画像の矩形を切って別の画像に貼る、label は paste 面積比で混ぜる。

<details>
<summary>📖 もっと詳しく</summary>

Yun 2019。Mixup より自然な合成画像、object localization のような spatial 情報を壊さない。

落とし穴 (interview trap): sampled λ じゃなく、画像境界での clipping 後の **実面積** から λ を再計算してから返す — 矩形が画像端で切れることがあるため。

</details>

### area-based λ (interview trap)
矩形はランダムな中心位置に置かれて、**画像境界でクリップされる**ことがある。なので Beta(α,α) からサンプルした λ と、**実際の**面積比は一致しない。clipping 後の実面積から λ を再計算してから return すること — interview で grill される所。


### Signature
```python
def cutmix(x, y, alpha=1.0):
    # x: (B, C, H, W) image batch
    # y: (B,) class indices
    # alpha: Beta distribution parameter
    # Returns: (x_mixed, y_a, y_b, lam)
    #   lam = 1 - (actual cut area) / (H*W) — recomputed after clipping
```

### Rules
- `λ ~ Beta(α, α)` をサンプル、cut size は `cut_h = int(H * sqrt(1 - λ))`, 同様に `cut_w`
- 中心 `(cy, cx)` を `[0, H) × [0, W)` から uniform サンプル
- 矩形を画像 bounds で clip: `y1 = max(0, cy - cut_h//2)` など
- `x[perm, :, y1:y2, x1:x2]` を `x_mixed` の同じ slice にコピー
- **λ を再計算** `1 - (y2-y1)*(x2-x1) / (H*W)` してから return

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def cutmix(x, y, alpha=1.0):
    pass  # lam サンプル、cut size 導出、中心サンプリング、画像 bounds で clip、
          # x[perm] を paste、最後に実面積から lam を再計算

In [ ]:
import torch
torch.manual_seed(0)
x = torch.zeros(2, 1, 16, 16)
x[1] = 1.0
y = torch.tensor([0, 1])
x_mix, y_a, y_b, lam = cutmix(x, y, alpha=1.0)
print('lam =', round(lam, 3))
print('Ones in sample 0:', (x_mix[0] == 1).sum().item(), '(should be (1-lam)*256 if perm swapped)')

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("cutmix")